# Exercise 6: a neural language model

A language model predicts the next word from its context. In this exercise,
we learn word representations and use the previous three words to predict
a distribution over the vocabulary.


### Model

Each word has a learned vector. The context is represented by concatenating
its word vectors in order:

$$x=\operatorname{concat}(\text{context word vectors}),\qquad
h=\tanh(xW_1+b_1),\qquad z=hW_2+b_2.$$

The scores $z$ define a distribution over next words:

$$p_j=\frac{e^{z_j}}{\sum_k e^{z_k}},\qquad
L=-\frac{1}{B}\sum_{i=1}^{B}\log p_{i,y_i}.$$

| Tensor | Shape | Meaning |
|---|---|---|
| `X` | `(B, C)` | Context word IDs |
| `y` | `(B,)` | Next-word IDs |
| `E` | `(V, D)` | Word vectors |
| `W1`, `b1` | `(C*D, H)`, `(H,)` | Hidden-layer parameters |
| `W2`, `b2` | `(H, V)`, `(V,)` | Output-layer parameters |
| `z` | `(B, V)` | Next-word scores |

`B` is batch size, `C` context length, `V` vocabulary size, `D` embedding
dimension, and `H` hidden width. Word IDs identify rows, not numerical features.

### Corpus

We use a small collection of template sentences, split into training,
validation, and test sets. The vocabulary contains training words only.
Sentences in different splits may share short contexts.

`<bos>` marks the beginning of a sentence, `<eos>` its end, and `<unk>`
an unknown word.

In [ ]:
import math
import random
from itertools import product

import torch

torch.manual_seed(42)
torch.set_num_threads(2)
rng = random.Random(42)
CONTEXT = 3

people = ["teacher", "student", "writer", "reader", "child", "friend"]
texts = ["book", "story", "poem", "letter", "article", "novel"]
animals = ["cat", "dog", "rabbit", "horse", "bird"]
foods = ["food", "fruit", "bread", "rice", "carrot"]
places = ["in the library", "in the garden", "at the school", "at the house"]
sentences = [
    f"the {person} {verb} the {text} {rng.choice(places)}"
    for person, verb, text in product(people, ["reads", "writes", "likes"], texts)
]
sentences += [
    f"the {animal} {verb} the {food} {rng.choice(places)}"
    for animal, verb, food in product(animals, ["eats", "finds", "likes"], foods)
]
rng.shuffle(sentences)
train_end = int(0.70 * len(sentences))
val_end = int(0.85 * len(sentences))
train_sentences = sentences[:train_end]
val_sentences = sentences[train_end:val_end]
test_sentences = sentences[val_end:]
words = ["<unk>", "<bos>", "<eos>"] + sorted({
    word for sentence in train_sentences for word in sentence.split()
})
word_to_id = {word: i for i, word in enumerate(words)}
V = len(words)

print("Example training sentences:")
for sentence in train_sentences[:3]:
    print(sentence)
print("Vocabulary:", words)
print("Number of sentences:", len(train_sentences), len(val_sentences), len(test_sentences))

## Task 1: Contexts and targets

Implement `make_examples` to return integer tensors of fixed-length contexts
and their next-word targets. Include sentence boundaries, pad initial contexts
with `<bos>`, and map unseen words to `<unk>`. Contexts must stay within a sentence.

**But why?** Raw data rarely arrives as model inputs and labels. Building these pairs correctly prevents a model from seeing the answer in its input and prepares it for sentence boundaries and unfamiliar words.


In [ ]:
def make_examples(sentences, word_to_id, context_size=CONTEXT):
    ### Your Solution starts here ###
    raise NotImplementedError
    ### Your solution ends here ###


**Why this check?** A tiny sentence lets us verify every context and target by hand. Checking boundaries and unknown words catches data mistakes before they affect training.


In [ ]:
mini_words = ["<unk>", "<bos>", "<eos>", "the", "cat", "eats"]
mini_vocab = {word: i for i, word in enumerate(mini_words)}
mini_X, mini_y = make_examples(["the cat eats"], mini_vocab, context_size=2)
assert mini_X.tolist() == [[1, 1], [1, 3], [3, 4], [4, 5]]
assert mini_y.tolist() == [3, 4, 5, 2]
_, unknown_y = make_examples(["the fox"], mini_vocab, context_size=2)
assert unknown_y.tolist() == [3, 0, 2]
assert mini_X.dtype == mini_y.dtype == torch.long
for context, target in zip(mini_X.tolist(), mini_y.tolist()):
    print([mini_words[i] for i in context], "->", mini_words[target])

X_train, y_train = make_examples(train_sentences, word_to_id)
X_val, y_val = make_examples(val_sentences, word_to_id)
print("Training shapes:", X_train.shape, y_train.shape)

## Task 2: Model parameters

Implement `make_parameters` to return a dictionary containing the five
learnable tensors in the model. Initialize word vectors and weights with
small random values and biases with zeros.

**But why?** Training needs adjustable numbers in which to store what the model learns. Random initialization gives hidden units different starting points, so they can learn different patterns.


In [ ]:
def make_parameters(vocab_size=V, context_size=CONTEXT, dim=8, hidden_size=24, seed=42):
    torch.manual_seed(seed)
    ### Your Solution starts here ###
    raise NotImplementedError
    ### Your solution ends here ###


**Why this check?** The parameter shapes must match the model, and every parameter must track gradients. These checks catch setup errors before the first forward pass.


In [ ]:
parameters = make_parameters()
expected_shapes = {
    "E": (V, 8), "W1": (CONTEXT * 8, 24), "b1": (24,),
    "W2": (24, V), "b2": (V,),
}
for name, shape in expected_shapes.items():
    assert parameters[name].shape == shape
    assert parameters[name].requires_grad
    print(name, tuple(parameters[name].shape))

## Task 3: Context representations

Implement `context_vectors` to represent each context by the concatenation
of its word vectors, preserving word order. Return a tensor of shape `(B, C*D)`.

**But why?** Word IDs alone carry no useful notion of similarity. Learned vectors let a model share information across words, while preserving order helps distinguish phrases such as “dog bites person” and “person bites dog”.


In [ ]:
def context_vectors(X, E):
    ### Your Solution starts here ###
    raise NotImplementedError
    ### Your solution ends here ###


**Why this check?** The same words in a different order should produce different context vectors. This small example checks both the embedding lookup and the order of concatenation.


In [ ]:
# Rows represent cat, dog, book.
toy_E = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
toy_X = torch.tensor([[0, 2], [2, 0]])
vectors = context_vectors(toy_X, toy_E)
assert vectors.tolist() == [[1., 0., 1., 1.], [1., 1., 1., 0.]]
assert context_vectors(toy_X[:1], toy_E).shape == (1, 4)
print(vectors)

## Task 4: Forward pass

Implement `forward` to compute the model's next-word scores for a batch of
contexts. Return a tensor of shape `(B, V)`.

**But why?** Every deployed model needs a computation that turns inputs into predictions. Processing several examples together also makes efficient use of the hardware during training and inference.


In [ ]:
def forward(X, parameters):
    ### Your Solution starts here ###
    raise NotImplementedError
    ### Your solution ends here ###


**Why this check?** A tiny network has scores we can calculate by hand. Comparing against them checks the actual computation, while the batch check verifies the output shape.


In [ ]:
# A one-word context and tiny weights make this check calculable by hand.
tiny = {
    "E": torch.tensor([[1.], [2.]]),
    "W1": torch.tensor([[1.]]), "b1": torch.zeros(1),
    "W2": torch.tensor([[1., -1.]]), "b2": torch.tensor([0., 0.5]),
}
tiny_scores = forward(torch.tensor([[0]]), tiny)
expected = torch.tensor([[math.tanh(1), -math.tanh(1) + 0.5]])
assert torch.allclose(tiny_scores, expected)
assert forward(X_train[:2], parameters).shape == (2, V)
print("Two contexts give scores with shape:", forward(X_train[:2], parameters).shape)

## Task 5: Prediction loss

Implement `log_probabilities` and `prediction_loss` to compute the
log-probabilities and the mean negative log-probability of the target words.
The computation should remain numerically stable for large scores and
support automatic differentiation.

**But why?** A loss turns prediction quality into a quantity that training can improve. Confident mistakes receive a larger penalty, and numerical stability keeps large scores from breaking the computation.


In [ ]:
def log_probabilities(logits):
    ### Your Solution starts here ###
    raise NotImplementedError
    ### Your solution ends here ###


def prediction_loss(logits, targets):
    ### Your Solution starts here ###
    raise NotImplementedError
    ### Your solution ends here ###


**Why this check?** Equal scores should give equal probabilities, and confident mistakes should cost more than correct predictions. Large scores also check that the calculation stays numerically stable.


In [ ]:
uniform = log_probabilities(torch.zeros(2, 3)).exp()
assert torch.allclose(uniform, torch.ones(2, 3) / 3)
uniform_loss = prediction_loss(torch.zeros(2, 3), torch.tensor([0, 2]))
assert abs(uniform_loss.item() - math.log(3)) < 1e-6
correct = prediction_loss(torch.tensor([[4., 0., 0.]]), torch.tensor([0]))
wrong = prediction_loss(torch.tensor([[0., 4., 0.]]), torch.tensor([0]))
assert correct < uniform_loss < wrong
large = torch.tensor([[1000., 1001., 999.]])
assert torch.allclose(log_probabilities(large).exp().sum(dim=1), torch.ones(1))
print("Correct, uncertain, wrong losses:", correct.item(), uniform_loss.item(), wrong.item())

## Task 6: Gradients

For $L=(w-3)^2$ at $w=1$, compare the gradient from automatic differentiation
with the analytical derivative. Investigate what happens to the stored
gradient when the loss is computed and differentiated a second time.

**But why?** Neural networks can contain millions of parameters, making manual differentiation impractical. Automatic differentiation computes their gradients, and understanding accumulation helps avoid unintended updates across batches.


In [ ]:
w = torch.tensor(1.0, requires_grad=True)

### Your Solution starts here ###
raise NotImplementedError
### Your solution ends here ###
# The analytical derivative checks that the first backward pass is correct.
assert w.grad.item() == -4.0

### Your Solution starts here ###
raise NotImplementedError
### Your solution ends here ###
# A second derivative contribution checks that gradients accumulate rather than replace the old value.
assert w.grad.item() == -8.0

## Task 7: A learning step

Implement `train_step` to perform one gradient-descent update on all model
parameters, using only the current batch's gradients. Return the loss before
the update as a Python number.

**But why?** A model learns when its parameters change in response to prediction errors. This update is the basic operation behind training systems for tasks such as image recognition, translation, and forecasting.


In [ ]:
def train_step(parameters, X, targets, learning_rate):
    ### Your Solution starts here ###
    raise NotImplementedError
    ### Your solution ends here ###


**Why this check?** A small update should reduce the loss on this batch, and the reported loss should match its value before the update. This catches a reversed update direction or a misleading return value.


In [ ]:
probe = make_parameters()
batch_X = X_train[:16]
batch_y = y_train[:16]
with torch.no_grad():
    before = prediction_loss(forward(batch_X, probe), batch_y).item()
reported = train_step(probe, batch_X, batch_y, learning_rate=0.1)
with torch.no_grad():
    after = prediction_loss(forward(batch_X, probe), batch_y).item()
assert abs(reported - before) < 1e-6
assert after < before
print("One small update:", before, "->", after)

## Task 8: Training

Implement `fit` to train for the requested number of full-batch updates and
return the loss history. Compare training and validation loss before and
after learning.

**But why?** One update is rarely enough to learn useful patterns. Tracking training and validation loss over repeated updates helps reveal whether the model is learning patterns that also help on unseen data.


In [ ]:
def fit(parameters, X, targets, updates, learning_rate):
    ### Your Solution starts here ###
    raise NotImplementedError
    ### Your solution ends here ###


**Why this check?** The loss history should contain one entry per update, and repeated updates should improve the training loss. Comparing validation loss also helps us see whether learning carries over to unseen sentences.


In [ ]:
trained_parameters = make_parameters()
with torch.no_grad():
    initial_train_loss = prediction_loss(forward(X_train, trained_parameters), y_train).item()
    initial_val_loss = prediction_loss(forward(X_val, trained_parameters), y_val).item()

loss_history = fit(trained_parameters, X_train, y_train, updates=400, learning_rate=0.5)
for update in [0, 9, 99, 399]:
    print(f"Before update {update + 1}: training loss = {loss_history[update]:.3f}")

with torch.no_grad():
    final_train_loss = prediction_loss(forward(X_train, trained_parameters), y_train).item()
    final_val_loss = prediction_loss(forward(X_val, trained_parameters), y_val).item()
assert len(loss_history) == 400
assert abs(loss_history[0] - initial_train_loss) < 1e-6
assert final_train_loss < initial_train_loss
print(f"Training loss: {initial_train_loss:.3f} -> {final_train_loss:.3f}")
print(f"Validation loss: {initial_val_loss:.3f} -> {final_val_loss:.3f}")

## Task 9: Next-word prediction

Implement `predict_next` to return a vocabulary-sized probability vector
for a text prompt, using its last `CONTEXT` words. Handle short prompts and
unknown words consistently with the training examples, and ignore case.

**But why?** Users provide text, not tensors. Prediction code connects a trained model to applications such as autocomplete and must process new inputs consistently with training.


In [ ]:
def predict_next(text, parameters):
    ### Your Solution starts here ###
    raise NotImplementedError
    ### Your solution ends here ###


**Why this check?** Predictions should form a valid probability distribution. Equivalent prompts check that padding short inputs and truncating long ones match the context used during training.


In [ ]:
for prompt in ["teacher reads the", "rabbit eats the", "teacher eats the"]:
    probabilities = predict_next(prompt, trained_parameters)
    assert probabilities.shape == (V,)
    assert abs(probabilities.sum().item() - 1.0) < 1e-5
    next_id = probabilities.argmax().item()
    print(prompt, "->", words[next_id], f"(probability {probabilities[next_id].item():.3f})")

assert torch.allclose(
    predict_next("cat", trained_parameters),
    predict_next("<bos> <bos> cat", trained_parameters),
)
assert torch.allclose(
    predict_next("extra words teacher reads the", trained_parameters),
    predict_next("teacher reads the", trained_parameters),
)

## Task 10: Evaluation

Compute test accuracy and perplexity, defined as $\exp(L)$. Compare the
model with the uniform baseline and examine some predictions. Explain why
a plausible next word can still count as an incorrect prediction.

**But why?** A model can fit its training data without working well on new examples. Testing against a simple baseline helps judge its usefulness, while inspecting individual predictions reveals limitations that a single score can hide.


In [ ]:
X_test, y_test = make_examples(test_sentences, word_to_id)
with torch.no_grad():
    ### Your Solution starts here ###
    raise NotImplementedError
    ### Your solution ends here ###

# These checks catch mismatched predictions and invalid metric values before we interpret the test results.
assert predicted_ids.shape == y_test.shape
assert 0.0 <= test_accuracy.item() <= 1.0
assert math.isfinite(test_perplexity) and test_perplexity >= 1.0
print(f"Test accuracy = {test_accuracy.item():.3f}")
print(f"Test perplexity = {test_perplexity:.3f}, uniform baseline = {V}")
for i in [2, 3, 4, 5]:
    context = " ".join(words[j] for j in X_test[i].tolist())
    prediction = words[predicted_ids[i].item()]
    observed = words[y_test[i].item()]
    print(f"{context} -> predicted: {prediction}, observed: {observed}")

### Reference

Bengio, Ducharme, Vincent, and Jauvin (2003).
*A Neural Probabilistic Language Model*. JMLR 3, 1137–1155.
This model omits the optional direct input-to-output connection.